# Tech Challenge - An??lise e Predi????o de Atrasos de Voos nos EUA

## 1. Introdu????o

Este projeto aplica um pipeline completo de Ci??ncia de Dados para analisar atrasos de voos e construir modelos de Machine Learning para prever se um voo vai atrasar ou n??o.

### Objetivos
- Entender padr??es de atraso por aeroporto, companhia a??rea e hor??rio;
- Criar vari??veis derivadas para enriquecer a modelagem;
- Treinar e comparar modelos supervisionados;
- Aplicar clusteriza????o para identificar perfis de voos.

### Bibliotecas utilizadas
- `pandas`
- `numpy`
- `matplotlib`
- `seaborn`
- `scikit-learn`

In [ ]:
# 2. Carregamento de bibliotecas
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.cluster import KMeans

sns.set_theme(style='whitegrid', palette='viridis')
pd.set_option('display.max_columns', 200)

## 2. Carregamento dos dados

Vamos utilizar um dataset p??blico de voos dos EUA (`nycflights13`).

> Observa????o: os voos partem de aeroportos de Nova York para diversos destinos nos EUA. Isso atende ao objetivo do desafio, mas a cobertura nacional completa ?? uma limita????o que ser?? discutida ao final.

In [ ]:
# 3. Leitura dos dados
# Tenta carregar um arquivo local; se n??o existir, faz download de fonte p??blica.
local_path = 'flights.csv'
public_urls = [
    'https://raw.githubusercontent.com/ismayc/nycflights13/master/data-raw/flights.csv',
    'https://raw.githubusercontent.com/tidyverse/nycflights13/master/data-raw/flights.csv'
]

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
    source_used = f'Arquivo local: {local_path}'
else:
    df = None
    source_used = None
    for url in public_urls:
        try:
            df = pd.read_csv(url)
            source_used = f'URL p??blica: {url}'
            break
        except Exception:
            continue

if df is None:
    raise RuntimeError(
        'N??o foi poss??vel carregar o dataset. '        'Baixe manualmente um CSV de voos e salve como flights.csv na mesma pasta.'
    )

print('Fonte utilizada ->', source_used)
print('Dimens??es do dataset:', df.shape)

## 3. Explora????o dos dados (EDA)

In [ ]:
# Estrutura inicial
display(df.head())
print('
Informa????es gerais:')
df.info()

In [ ]:
# Estat??sticas descritivas
print('Estat??sticas num??ricas:')
display(df.describe(include=[np.number]).T)

print('Estat??sticas de vari??veis categ??ricas:')
display(df.describe(include=['object']).T)

In [ ]:
# Valores ausentes
missing_abs = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_df = pd.DataFrame({'faltantes': missing_abs, 'percentual_%': missing_pct.round(2)})

print('Colunas com valores ausentes:')
display(missing_df[missing_df['faltantes'] > 0])

In [ ]:
# Visualiza????o da distribui????o de atraso de chegada (ARR_DELAY)
plt.figure(figsize=(10, 5))
arr_delay_clean = df['arr_delay'].dropna()
arr_delay_clip = arr_delay_clean.clip(lower=-60, upper=240)
sns.histplot(arr_delay_clip, bins=80, kde=True)
plt.title('Distribui????o do atraso de chegada (recortada entre -60 e 240 minutos)')
plt.xlabel('Atraso de chegada (min)')
plt.ylabel('Frequ??ncia')
plt.show()

In [ ]:
# Atraso m??dio por aeroporto de origem
atraso_por_origem = (
    df.groupby('origin', as_index=False)['arr_delay']
      .mean()
      .sort_values('arr_delay', ascending=False)
)

plt.figure(figsize=(8, 4))
sns.barplot(data=atraso_por_origem, x='origin', y='arr_delay')
plt.title('Atraso m??dio de chegada por aeroporto de origem')
plt.xlabel('Aeroporto de origem')
plt.ylabel('Atraso m??dio (min)')
plt.show()

display(atraso_por_origem)

In [ ]:
# Atraso m??dio por companhia a??rea
atraso_por_companhia = (
    df.groupby('carrier', as_index=False)['arr_delay']
      .mean()
      .sort_values('arr_delay', ascending=False)
)

plt.figure(figsize=(14, 5))
sns.barplot(data=atraso_por_companhia, x='carrier', y='arr_delay')
plt.title('Atraso m??dio de chegada por companhia a??rea')
plt.xlabel('Companhia a??rea (c??digo)')
plt.ylabel('Atraso m??dio (min)')
plt.xticks(rotation=45)
plt.show()

display(atraso_por_companhia.head(10))

In [ ]:
# Padr??o por hor??rio de partida agendado (hora)
atraso_por_hora = (
    df.groupby('hour', as_index=False)['arr_delay']
      .mean()
      .sort_values('hour')
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=atraso_por_hora, x='hour', y='arr_delay', marker='o')
plt.title('Atraso m??dio de chegada por hora prevista de partida')
plt.xlabel('Hora prevista de partida')
plt.ylabel('Atraso m??dio (min)')
plt.show()

display(atraso_por_hora)

## 4. Limpeza e tratamento

Nesta etapa, vamos:
- tratar tipos de dados;
- criar vari??veis de data para an??lises temporais;
- preparar os dados para modelagem.

In [ ]:
# C??pia de trabalho
data = df.copy()

# Normaliza????o de nomes para letras mai??sculas (conven????o visual no notebook)
data.columns = [c.upper() for c in data.columns]

# Cria coluna de data completa
data['DATA_VOO'] = pd.to_datetime(
    data[['YEAR', 'MONTH', 'DAY']].rename(
        columns={'YEAR': 'year', 'MONTH': 'month', 'DAY': 'day'}
    ),
    errors='coerce'
)

# Dia da semana (0=Segunda, 6=Domingo)
data['DIA_SEMANA'] = data['DATA_VOO'].dt.dayofweek

# Tratamento de coluna de hora prevista (quando HOUR estiver ausente)
# Usa SCHED_DEP_TIME no formato HHMM para extrair hora.
if 'HOUR' in data.columns:
    data['HOUR'] = data['HOUR'].fillna((data['SCHED_DEP_TIME'] // 100).astype('float'))
else:
    data['HOUR'] = (data['SCHED_DEP_TIME'] // 100).astype('float')

# Mant??m apenas horas v??lidas entre 0 e 23
data.loc[~data['HOUR'].between(0, 23), 'HOUR'] = np.nan

print('Shape ap??s prepara????o inicial:', data.shape)
display(data.head())

## 5. Engenharia de vari??veis

### Vari??veis derivadas
- `PERIODO_DIA`: manh??, tarde e noite;
- `DIA_SEMANA`: j?? criado a partir da data;
- vari??vel alvo `ATRASO`, onde:
  - `1` se `ARR_DELAY > 0`
  - `0` caso contr??rio

In [ ]:
# Per??odo do dia com base na hora prevista de partida
# Regra simples e interpret??vel:
# 05-11: manh?? | 12-17: tarde | demais: noite

def map_periodo(h):
    if pd.isna(h):
        return np.nan
    h = int(h)
    if 5 <= h <= 11:
        return 'manha'
    if 12 <= h <= 17:
        return 'tarde'
    return 'noite'


data['PERIODO_DIA'] = data['HOUR'].apply(map_periodo)

# Vari??vel alvo
# Considera apenas voos com informa????o de atraso de chegada.
data['ATRASO'] = np.where(data['ARR_DELAY'] > 0, 1, 0)

display(data[['ARR_DELAY', 'ATRASO', 'HOUR', 'PERIODO_DIA', 'DIA_SEMANA']].head(10))
print('Propor????o da classe ATRASO=1:', round(data['ATRASO'].mean(), 4))

In [ ]:
# Visualiza????o: atraso por dia da semana
# Mapeamento para facilitar leitura
mapa_dia = {
    0: 'Seg', 1: 'Ter', 2: 'Qua', 3: 'Qui',
    4: 'Sex', 5: 'Sab', 6: 'Dom'
}

tmp = (
    data.dropna(subset=['ARR_DELAY'])
        .groupby('DIA_SEMANA', as_index=False)['ARR_DELAY']
        .mean()
        .sort_values('DIA_SEMANA')
)
tmp['DIA_LABEL'] = tmp['DIA_SEMANA'].map(mapa_dia)

plt.figure(figsize=(9, 4))
sns.barplot(data=tmp, x='DIA_LABEL', y='ARR_DELAY')
plt.title('Atraso m??dio de chegada por dia da semana')
plt.xlabel('Dia da semana')
plt.ylabel('Atraso m??dio (min)')
plt.show()

display(tmp)

## 6. Modelagem supervisionada

Vamos comparar dois modelos:
1. Regress??o Log??stica
2. Random Forest

### Estrat??gia
- Sele????o de features num??ricas e categ??ricas;
- Pipeline com imputa????o + OneHotEncoding + modelo;
- Separa????o treino/teste com `train_test_split`.

In [ ]:
# Sele????o de vari??veis
# Nota: DEP_DELAY ?? historicamente muito informativa para ARR_DELAY.
feature_cols = [
    'MONTH', 'DAY', 'DIA_SEMANA', 'HOUR',
    'DISTANCE', 'AIR_TIME', 'DEP_DELAY',
    'ORIGIN', 'DEST', 'CARRIER', 'PERIODO_DIA'
]

model_data = data[feature_cols + ['ATRASO']].copy()

# Remove linhas sem alvo v??lido (ARR_DELAY ausente pode distorcer)
# Como ATRASO foi criado com np.where, garantimos consist??ncia removendo casos onde ARR_DELAY era NaN
model_data = model_data.loc[data['ARR_DELAY'].notna()].copy()

X = model_data[feature_cols]
y = model_data['ATRASO']

categorical_features = ['ORIGIN', 'DEST', 'CARRIER', 'PERIODO_DIA']
numeric_features = [c for c in feature_cols if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print('Treino:', X_train.shape, '| Teste:', X_test.shape)
print('Taxa de atraso no treino:', round(y_train.mean(), 4))
print('Taxa de atraso no teste :', round(y_test.mean(), 4))

In [ ]:
# Pr??-processamento
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Modelos
log_reg_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=200, random_state=42))
])

rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
# Fun????o de avalia????o

def avaliar_modelo(nome_modelo, pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f'===== {nome_modelo} =====')
    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'F1-score : {f1:.4f}')
    print('
Relat??rio de classifica????o:')
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de confus??o - {nome_modelo}')
    plt.xlabel('Predito')
    plt.ylabel('Real')
    plt.show()

    return {
        'Modelo': nome_modelo,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    }

In [ ]:
# Treino e avalia????o dos modelos
result_log = avaliar_modelo('Logistic Regression', log_reg_pipeline, X_train, X_test, y_train, y_test)
result_rf = avaliar_modelo('Random Forest', rf_pipeline, X_train, X_test, y_train, y_test)

resultados_supervisionado = pd.DataFrame([result_log, result_rf]).sort_values('F1', ascending=False)
print('Compara????o de modelos (ordenado por F1):')
display(resultados_supervisionado)

## 7. Modelagem n??o supervisionada (KMeans)

Objetivo: identificar perfis de voos com base em:
- `DISTANCE`
- `DEP_DELAY`
- `ARR_DELAY`

Passos:
- tratar nulos;
- padronizar vari??veis (StandardScaler);
- aplicar KMeans;
- visualizar clusters.

In [ ]:
cluster_vars = ['DISTANCE', 'DEP_DELAY', 'ARR_DELAY']
cluster_data = data[cluster_vars].dropna().copy()

# Opcional: reduz impacto de outliers extremos para melhorar visualiza????o
cluster_data['DEP_DELAY'] = cluster_data['DEP_DELAY'].clip(-30, 300)
cluster_data['ARR_DELAY'] = cluster_data['ARR_DELAY'].clip(-60, 300)

scaler = StandardScaler()
X_cluster = scaler.fit_transform(cluster_data)

# Defini????o de k
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_data['CLUSTER'] = kmeans.fit_predict(X_cluster)

print('Quantidade por cluster:')
display(cluster_data['CLUSTER'].value_counts().sort_index())

In [ ]:
# Visualiza????o dos clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=cluster_data.sample(min(20000, len(cluster_data)), random_state=42),
    x='DEP_DELAY',
    y='ARR_DELAY',
    hue='CLUSTER',
    palette='Set2',
    alpha=0.6,
    s=30
)
plt.title('Clusters de voos (DEP_DELAY x ARR_DELAY)')
plt.xlabel('Atraso na partida (min)')
plt.ylabel('Atraso na chegada (min)')
plt.legend(title='Cluster')
plt.show()

In [ ]:
# Interpreta????o dos clusters via centr??ides (em escala original)
centroids_scaled = kmeans.cluster_centers_
centroids = scaler.inverse_transform(centroids_scaled)
centroids_df = pd.DataFrame(centroids, columns=cluster_vars)
centroids_df['CLUSTER'] = range(len(centroids_df))
centroids_df = centroids_df[['CLUSTER'] + cluster_vars]

print('Centr??ides dos clusters (escala original):')
display(centroids_df.round(2))

## 8. Resultados

Nesta se????o, respondemos ??s principais perguntas de neg??cio com base nas an??lises.

In [ ]:
# Top aeroportos e companhias com maior atraso m??dio
top_aeroportos = (
    data.groupby('ORIGIN', as_index=False)['ARR_DELAY']
        .mean()
        .sort_values('ARR_DELAY', ascending=False)
)

top_companhias = (
    data.groupby('CARRIER', as_index=False)['ARR_DELAY']
        .mean()
        .sort_values('ARR_DELAY', ascending=False)
)

print('Aeroportos com maior atraso m??dio:')
display(top_aeroportos)

print('Companhias com maior atraso m??dio (top 10):')
display(top_companhias.head(10))

print('Comparativo final dos modelos:')
display(resultados_supervisionado)

## 9. Conclus??es

Com base nos resultados obtidos:

1. **Quais aeroportos t??m mais atrasos?**
   - O ranking de atraso m??dio por `ORIGIN` mostra quais aeroportos de origem concentram maior atraso.

2. **Que fatores aumentam a chance de atraso?**
   - `DEP_DELAY` (atraso na partida) ?? um dos fatores mais fortes para aumento de `ARR_DELAY`.
   - Hor??rio (`HOUR`/`PERIODO_DIA`), companhia (`CARRIER`) e rota (`ORIGIN`/`DEST`) tamb??m influenciam o comportamento.

3. **?? poss??vel prever atrasos com base no hist??rico?**
   - Sim. Os modelos supervisionados apresentam capacidade de classifica????o ??til.
   - O melhor modelo pode ser identificado comparando `Accuracy`, `Precision`, `Recall` e `F1`.

## 10. Limita????es

- O dataset usado representa voos partindo de Nova York, n??o todos os aeroportos dos EUA.
- H?? aus??ncia de vari??veis externas importantes (clima, tr??fego a??reo, manuten????o).
- Modelos podem sofrer com desbalanceamento de classes dependendo do recorte temporal.
- M??tricas podem variar com tuning de hiperpar??metros e valida????o cruzada.

## 11. Pr??ximos passos (Melhorias futuras)

- Incluir dados meteorol??gicos por aeroporto e hor??rio.
- Testar modelos mais avan??ados, como **XGBoost**, LightGBM e CatBoost.
- Aplicar valida????o temporal (time-based split) para cen??rios mais realistas.
- Implementar explicabilidade com SHAP para entender drivers de atraso.
- Criar dashboard interativo (Streamlit/Power BI) para acompanhamento operacional.

In [ ]:
# C??lula opcional: salvar resultados em CSV para apresenta????o
resultados_supervisionado.to_csv('comparacao_modelos.csv', index=False)
centroids_df.to_csv('centroides_clusters.csv', index=False)
print('Arquivos exportados: comparacao_modelos.csv e centroides_clusters.csv')